In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# Chargement du dataset pour cette session (fichier précédemment utilisé)
df = pd.read_csv("/content/drive/MyDrive/ColabNotebooks/detection_anomalie_transaction_bancaire/creditcard.csv")
df = df.drop_duplicates()

X = df.drop('Class', axis=1)
y = df['Class']

# Définir les grilles de paramètres à tester
param_grid = {
    "n_estimators": [100, 200],
    "max_samples": [0.6, 0.75, 1.0],
    "max_features": [0.5, 1.0],
    "bootstrap": [False, True],
    "contamination": [0.003]  # Fixé pour simplifier l’analyse
}

# Tester toutes les combinaisons
results = []
from itertools import product

for n_est, max_samp, max_feat, boot, cont in product(
    param_grid["n_estimators"],
    param_grid["max_samples"],
    param_grid["max_features"],
    param_grid["bootstrap"],
    param_grid["contamination"]
):
    iso = IsolationForest(
        n_estimators=n_est,
        max_samples=max_samp,
        max_features=max_feat,
        contamination=cont,
        bootstrap=boot,
        random_state=42
    )
    preds = iso.fit_predict(X)
    df['anomaly_pred'] = pd.Series(preds, index=df.index).map({1: 0, -1: 1})

    cm = confusion_matrix(y, df['anomaly_pred'])
    TN, FP, FN, TP = cm.ravel()
    recall = TP / (TP + FN) if TP + FN > 0 else 0
    precision = TP / (TP + FP) if TP + FP > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    results.append({
        "n_estimators": n_est,
        "max_samples": max_samp,
        "max_features": max_feat,
        "bootstrap": boot,
        "contamination": cont,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Recall": round(recall, 4),
        "Precision": round(precision, 4),
        "F1_score": round(f1, 4)
    })

results_df = pd.DataFrame(results)

In [ ]:
print("results_df : \n", results_df)

results_df : 
     n_estimators  max_samples  max_features  bootstrap  contamination   TP  \
0            100         0.60           0.5      False          0.003  230   
1            100         0.60           0.5       True          0.003  232   
2            100         0.60           1.0      False          0.003  231   
3            100         0.60           1.0       True          0.003  212   
4            100         0.75           0.5      False          0.003  236   
5            100         0.75           0.5       True          0.003  222   
6            100         0.75           1.0      False          0.003  229   
7            100         0.75           1.0       True          0.003  215   
8            100         1.00           0.5      False          0.003  228   
9            100         1.00           0.5       True          0.003  230   
10           100         1.00           1.0      False          0.003  209   
11           100         1.00           1.0      